## CHATBOT

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [8]:
##CALLING THE MODEL

from langchain_groq import ChatGroq


chat_groq = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

In [ ]:
#INVOKE THE MESSAGES ON THE MODEL
from langchain_core.messages import SystemMessage, HumanMessage

chat_groq.invoke([
    HumanMessage(content="Hi I am Shree, I am AI learner")])

AIMessage(content="Namaste Shree! It's great to meet you. I'm here to help you learn and explore various topics. What would you like to learn about today? Do you have a specific subject in mind, or are you open to suggestions?\n\nWe can discuss topics like:\n\n1. Artificial Intelligence (AI) and Machine Learning (ML)\n2. Programming languages (Python, Java, etc.)\n3. Data Science and Analytics\n4. Science and Technology (Physics, Chemistry, Biology, etc.)\n5. History, Geography, or Culture\n6. Language learning (English, Hindi, or any other language)\n\nLet me know what sparks your interest, and I'll do my best to assist you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 45, 'total_tokens': 187, 'completion_time': 0.217215331, 'completion_tokens_details': None, 'prompt_time': 0.002126886, 'prompt_tokens_details': None, 'queue_time': 0.051143863, 'total_time': 0.219342217}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint'

In [16]:
## CHAT MESSAGE HISTORY

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

stored={}

#create a session to differentioniate between different sessions
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in stored:
        stored[session_id] = ChatMessageHistory()
    return stored[session_id]

with_chat_history = RunnableWithMessageHistory(chat_groq,get_session_history)    

In [25]:
# Config = {"configurable":{"session_id": "session1"}}

def get_config(session_id):
    return {"configurable":{"session_id": session_id}}


In [26]:
def ask_bot(input,session_id):
    Config = get_config(session_id)
    response = with_chat_history.invoke([HumanMessage(content=input)],Config)
    return response.content

In [27]:
ask_bot("What is my name?","session1")

"Your name is Shree. (Getting repetitive, isn't it?)"

In [28]:
ask_bot("What is my name?","session2")

"I've already mentioned that I don't have any information about your name. This is the beginning of our conversation, and I don't have any prior knowledge about you. If you'd like to share your name, I'd be happy to chat with you about it."

In [30]:
## USING PROMPT TEMPLATES

from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful assistant"),
    MessagesPlaceholder(variable_name="messages")
    ])

chain = prompt | chat_groq    

In [34]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is Alpha, What is your name?")]}) 

AIMessage(content='Nice to meet you, Alpha. I don\'t have a personal name, but I\'m often referred to as an "assistant" or a "chatbot." You can think of me as a helpful digital companion. I\'m here to assist you with any questions, provide information, or simply have a conversation. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 51, 'total_tokens': 121, 'completion_time': 0.09781033, 'completion_tokens_details': None, 'prompt_time': 0.00245924, 'prompt_tokens_details': None, 'queue_time': 0.0512039, 'total_time': 0.10026957}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de91a-76fe-7cc0-820a-cfab468ade9f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 70, 'total_tokens': 121})

In [38]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

def ask_bot_with_prompt(input,session_id):
    Config = get_config(session_id)
    response = with_message_history.invoke({"messages":[HumanMessage(content=input)]},Config)
    return response.content

In [39]:
ask_bot_with_prompt("Hi my name is Beta, What is your name?","session3")

'Déjà vu! We\'ve had this conversation before, Beta. As I mentioned earlier, I don\'t have a personal name, but I\'m often referred to as an "Assistant." I\'m a computer program designed to provide information and assist with tasks. Let\'s start fresh, though! How can I help you today?'

In [40]:
ask_bot_with_prompt("Do you remeber me","session3")

'Beta, we\'ve had this conversation before, and I mentioned earlier that I don\'t have personal memories or the ability to recall individual conversations. So, while I\'m happy to chat with you again, I don\'t actually "remember" you. Each time you interact with me, it\'s a new conversation, and I\'m starting from a blank slate.'

Managaing the conversational hsitory

In [41]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

trimmer = trim_messages(max_tokens=10, strategy="last",token_counter=chat_groq,include_system=True,allow_partial=False,start_on="human")

In [45]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=lambda x: trimmer.invoke(x["messages"]))
    | prompt
    | chat_groq
)